In [21]:
!pip install gliner torch pandas openpyxl

In [22]:
import torch
import pandas as pd
from gliner import GLiNER
from typing import List, Dict


In [23]:
DATA_PATH = "/content/sample_data/trump_tweets_sota_classified(1).xlsx"

df = pd.read_excel(DATA_PATH)

print("Columns:", df.columns.tolist())
print("Total rows:", len(df))


Columns: ['tweet_id', 'tweet_text', 'date', 'primary_asset_class', 'primary_token', 'primary_ticker', 'sentiment', 'sentiment_score', 'event_types', 'entities', 'signal_direction', 'signal_strength', 'confidence', 'latency_ms', 'layers_used']
Total rows: 255


In [24]:
df[["tweet_id", "tweet_text"]].head()


,tweet_id,tweet_text
0,f6b6bc55c8b5,washingtonexaminer.com/restori
1,362a7010b3c1,breitbart.com/politics/2025/06
2,65c19d0b7fb5,redstate.com/redstate-guest-ed
3,8f0d572a979f,foxnews.com/opinion/loeffler-t
4,707c1b8d12dd,nypost.com/2025/06/28/us-news/


In [25]:
df["tweet_text"] = df["tweet_text"].astype(str).str.strip()
df = df[df["tweet_text"] != ""]


In [61]:
model = GLiNER.from_pretrained("urchade/gliner_base")

model.eval()  # 🔴 CRITICAL
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded on {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:186: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded on cpu


In [63]:
NER_LABELS = [
    "person name like Putin or Biden",
    "country name like Ukraine or China",
    "organization name like Fed or OPEC",
    "commodity name like oil or gold",
    "currency name like dollar or euro"
]

In [71]:
import re

def clean_tweet(text: str) -> str:
    """Clean tweet text for better NER performance."""
    if not isinstance(text, str):
        return ""
    # Remove URLs (including bare domains like example.com/path)
    text = re.sub(r'https?://\S+', '', text)  # http:// or https://
    text = re.sub(r'www\.\S+', '', text)       # www.
    text = re.sub(r'\S+\.(com|org|net|gov|io|co|edu|info|biz|me)/\S*', '', text)  # bare domains with path
    text = re.sub(r'\S+\.(com|org|net|gov|io|co|edu|info|biz|me)\b', '', text)    # bare domains without path
    # Remove mentions but keep the name
    text = re.sub(r'@(\w+)', r'\1', text)
    # Remove hashtag symbol but keep the word
    text = re.sub(r'#(\w+)', r'\1', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_valid_tweet(text: str) -> bool:
    """Check if tweet has meaningful text content (not just a URL)."""
    if not isinstance(text, str):
        return False
    cleaned = clean_tweet(text)
    # Must have at least 10 chars of actual content after cleaning
    return len(cleaned) >= 10

def extract_entities(
    text: str,
    labels=NER_LABELS,
    threshold: float = 0.05
):
    if not isinstance(text, str) or not text.strip():
        return []

    entities = model.predict_entities(
        text=text,
        labels=labels,
        threshold=threshold,
        flat_ner=True  # Re-adding flat_ner=True for granular entities
    )

    # Debugging: Print raw predictions before filtering
    print(f"Raw predictions for '{text}': {entities}")

    return [
        {
            "entity": e["text"],
            "label": e["label"],
            "score": round(e["score"], 3)
        }
        for e in entities
    ]

In [72]:
# Check how many tweets are just URLs vs actual text
url_only_count = sum(1 for t in df["tweet_text"] if not is_valid_tweet(t))
valid_count = sum(1 for t in df["tweet_text"] if is_valid_tweet(t))

print(f"Total tweets: {len(df)}")
print(f"URL-only tweets (skipped): {url_only_count}")
print(f"Valid text tweets: {valid_count}")
print(f"\nExamples of URL-only tweets:")
for i, t in enumerate(df["tweet_text"]):
    if not is_valid_tweet(t):
        print(f"  - {str(t)[:60]}")
        if i > 4:
            break

Total tweets: 255
URL-only tweets (skipped): 50
Valid text tweets: 205

Examples of URL-only tweets:
  - washingtonexaminer.com/restori
  - breitbart.com/politics/2025/06
  - redstate.com/redstate-guest-ed
  - foxnews.com/opinion/loeffler-t
  - nypost.com/2025/06/28/us-news/
  - whitehouse.gov/articles/2025/0


In [74]:
# Find a good sample tweet with actual text content
print("Examples of valid tweets with text content:\n")
for i, row in df.iterrows():
    text = row["tweet_text"]
    if is_valid_tweet(text):
        cleaned = clean_tweet(text)
        print(f"Tweet {i}:")
        print(f"  Original: {text[:100]}...")
        print(f"  Cleaned:  {cleaned[:100]}...")
        entities = extract_entities(text)
        if entities:
            print(f"  Entities:")
            for e in entities:
                print(f"    → {e['label']}: {e['entity']} ({e['score']})")
        else:
            print("  Entities: None found")
        print()
        if i > 15:  # Show first few valid examples
            break

Examples of valid tweets with text content:

Tweet 5:
  Original: I apologize for the long wait on the Faith Leaders Conference Call. AT&T ought to get its act togeth...
  Cleaned:  I apologize for the long wait on the Faith Leaders Conference Call. AT&T ought to get its act togeth...
Raw predictions for 'I apologize for the long wait on the Faith Leaders Conference Call. AT&T ought to get its act together. Please pass along the word to the tens of thousands of people who are there. We may have to resc...': [{'start': 37, 'end': 72, 'text': 'Faith Leaders Conference Call. AT&T', 'label': 'commodity name like oil or gold', 'score': 0.08294535428285599}, {'start': 159, 'end': 201, 'text': 'people who are there. We may have to resc.', 'label': 'country name like Ukraine or China', 'score': 0.07102049887180328}]
  Entities:
    → commodity name like oil or gold: Faith Leaders Conference Call. AT&T (0.083)
    → country name like Ukraine or China: people who are there. We may have to resc. 

In [75]:
test_text = "Oil prices are high. Putin spoke about Ukraine. Dollar is strong."

print(f"Testing general text: {test_text}")
results_general_text = extract_entities(test_text)
if results_general_text:
    print("Found entities:")
    for entity in results_general_text:
        print(f"  - {entity['label']}: {entity['entity']} (Score: {entity['score']})")
else:
    print("No entities found for this general text.")

Testing general text: Oil prices are high. Putin spoke about Ukraine. Dollar is strong.
Raw predictions for 'Oil prices are high. Putin spoke about Ukraine. Dollar is strong.': [{'start': 0, 'end': 54, 'text': 'Oil prices are high. Putin spoke about Ukraine. Dollar', 'label': 'person name like Putin or Biden', 'score': 0.0978536531329155}]
Found entities:
  - person name like Putin or Biden: Oil prices are high. Putin spoke about Ukraine. Dollar (Score: 0.098)


In [76]:
specific_tweet_text = "I apologize for the long wait on the Faith Leaders Conference Call. AT&T ought to get its act together."
print(f"Testing tweet with very low threshold (0.1): {specific_tweet_text}")
results_low_threshold = extract_entities(specific_tweet_text, threshold=0.1)
if results_low_threshold:
    print("Found entities with 0.1 threshold:")
    for entity in results_low_threshold:
        print(f"  - {entity['label']}: {entity['entity']} (Score: {entity['score']})")
else:
    print("No entities found even with 0.1 threshold for this specific tweet.")

Testing tweet with very low threshold (0.1): I apologize for the long wait on the Faith Leaders Conference Call. AT&T ought to get its act together.
Raw predictions for 'I apologize for the long wait on the Faith Leaders Conference Call. AT&T ought to get its act together.': [{'start': 37, 'end': 72, 'text': 'Faith Leaders Conference Call. AT&T', 'label': 'country name like Ukraine or China', 'score': 0.1590319126844406}]
Found entities with 0.1 threshold:
  - country name like Ukraine or China: Faith Leaders Conference Call. AT&T (Score: 0.159)


In [79]:
sample = df.loc[0, "tweet_text"]
sample


'washingtonexaminer.com/restori'

In [80]:
extract_entities(sample)


Raw predictions for 'washingtonexaminer.com/restori': [{'start': 0, 'end': 30, 'text': 'washingtonexaminer.com/restori', 'label': 'commodity name like oil or gold', 'score': 0.13208290934562683}]


[{'entity': 'washingtonexaminer.com/restori',
  'label': 'commodity name like oil or gold',
  'score': 0.132}]

In [81]:
df["entities"] = df["tweet_text"].apply(extract_entities)


Raw predictions for 'washingtonexaminer.com/restori': [{'start': 0, 'end': 30, 'text': 'washingtonexaminer.com/restori', 'label': 'commodity name like oil or gold', 'score': 0.13208290934562683}]
Raw predictions for 'breitbart.com/politics/2025/06': [{'start': 0, 'end': 30, 'text': 'breitbart.com/politics/2025/06', 'label': 'person name like Putin or Biden', 'score': 0.13495050370693207}]
Raw predictions for 'redstate.com/redstate-guest-ed': [{'start': 0, 'end': 30, 'text': 'redstate.com/redstate-guest-ed', 'label': 'commodity name like oil or gold', 'score': 0.13462543487548828}]
Raw predictions for 'foxnews.com/opinion/loeffler-t': [{'start': 0, 'end': 30, 'text': 'foxnews.com/opinion/loeffler-t', 'label': 'person name like Putin or Biden', 'score': 0.11655747890472412}]
Raw predictions for 'nypost.com/2025/06/28/us-news/': [{'start': 0, 'end': 30, 'text': 'nypost.com/2025/06/28/us-news/', 'label': 'person name like Putin or Biden', 'score': 0.13180170953273773}]
Raw predictions for 

In [82]:
df[["tweet_id", "tweet_text", "entities"]].head(5)


,tweet_id,tweet_text,entities
0,f6b6bc55c8b5,washingtonexaminer.com/restori,"[{'entity': 'washingtonexaminer.com/restori', ..."
1,362a7010b3c1,breitbart.com/politics/2025/06,"[{'entity': 'breitbart.com/politics/2025/06', ..."
2,65c19d0b7fb5,redstate.com/redstate-guest-ed,"[{'entity': 'redstate.com/redstate-guest-ed', ..."
3,8f0d572a979f,foxnews.com/opinion/loeffler-t,"[{'entity': 'foxnews.com/opinion/loeffler-t', ..."
4,707c1b8d12dd,nypost.com/2025/06/28/us-news/,"[{'entity': 'nypost.com/2025/06/28/us-news/', ..."


In [83]:
flat_entities = []

for _, row in df.iterrows():
    for ent in row["entities"]:
        flat_entities.append({
            "tweet_id": row["tweet_id"],
            "entity": ent["entity"].lower(),
            "label": ent["label"],
            "score": ent["score"]
        })

ner_df = pd.DataFrame(flat_entities)
ner_df.head(10)
ner_df


,tweet_id,entity,label,score
0,f6b6bc55c8b5,washingtonexaminer.com/restori,commodity name like oil or gold,0.132
1,362a7010b3c1,breitbart.com/politics/2025/06,person name like Putin or Biden,0.135
2,65c19d0b7fb5,redstate.com/redstate-guest-ed,commodity name like oil or gold,0.135
3,8f0d572a979f,foxnews.com/opinion/loeffler-t,person name like Putin or Biden,0.117
4,707c1b8d12dd,nypost.com/2025/06/28/us-news/,person name like Putin or Biden,0.132
...,...,...,...,...
505,fadeaf1d77b4,unyielding strength,commodity name like oil or gold,0.054
506,fadeaf1d77b4,"overwhelming force. time and again, our enemi...",organization name like Fed or OPEC,0.179
507,7a1386297599,youtube.com/live/447wkxyiijc?s,person name like Putin or Biden,0.137
508,68c3cb276bab,brad little is the strong and highly popular g...,commodity name like oil or gold,0.118


In [84]:
ner_df.groupby(["label", "entity"]) \
      .size() \
      .sort_values(ascending=False) \
      .head(30)


label                               entity                                                         
country name like Ukraine or China  iran                                                               9
                                    los angeles                                                        5
person name like Putin or Biden     donald j. trump, president of the united states                    3
commodity name like oil or gold     u.s. senate                                                        2
                                    democrats                                                          2
                                    u.s. army                                                          2
                                    administration                                                     2
country name like Ukraine or China  canada                                                             2
commodity name like oil or gold     fed                                                                2
                                    united states                                                      2
organization name like Fed or OPEC  national guard                                                     2
country name like Ukraine or China  israel and iran                                                    2
commodity name like oil or gold     china                                                              2
organization name like Fed or OPEC  republicans                                                        2
person name like Putin or Biden     luck. i’ll see you all in d.c                                      2
organization name like Fed or OPEC  pride—because you are the righteous sword of american justice      1
                                    peace talks                                                        1
                                    parliamentarian), should not be allowed to hurt the republicans    1
                                    overwhelming force. time and again, our enemi...                   1
                                    remember, you still have to get reelected                          1
                                    republican                                                         1
                                    u.s. army                                                          1
                                    u.s., also. it was my great honor                                  1
                                    rapists, murderers, and terrorists. this tsunami                   1
                                    operation. in a certain and very ironic way, that perfect          1
                                    one of the easiest, yet most prestigious, jobs in america          1
                                    oil from iran                                                      1
                                    united states army. but for the american people                    1
                                    not for now. but we don’t want missiles                            1
                                    news conferences                                                   1
dtype: int64